# Дополнительный этап: пакетный inference и API

Notebook повторно оценивает готовую test ranking table сохранённым CatBoost. Он не обучает ALS, Content-Based или CatBoost и не является обязательной частью учебного pipeline.

## Подключение проекта

**Что делаем:** определяем корень проекта.  
**Зачем:** одинаковые пути должны работать локально и в Colab.  
**Что получим:** `PROJECT_ROOT` и доступный пакет из `src`.

In [1]:
from pathlib import Path
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/fashion-recommender-system")
except ImportError:
    PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError(f"Не найдена папка src: {PROJECT_ROOT / 'src'}")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Корень проекта:", PROJECT_ROOT)

Корень проекта: <PROJECT_ROOT>


### Зависимости

**Что делаем:** устанавливаем requirements только в Colab.  
**Зачем:** локальное окружение не должно изменяться при каждом запуске.  
**Что получим:** готовые библиотеки для следующих ячеек.

In [2]:
import subprocess

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(PROJECT_ROOT / "requirements.txt")],
        check=True,
    )

### Импорты

**Что делаем:** подключаем pandas и функции загрузки/сохранения  
**Зачем:** в notebook нет training imports  
**Что получим:** минимальный inference-набор

In [3]:
import pandas as pd
from IPython.display import display

from fashion_recommender.persistence import (
    load_catboost_model, load_json, load_recommendations, save_recommendations,
)

### Пути к artifacts

**Что делаем:** задаём модель, ranking table и config files  
**Зачем:** batch demo полностью зависит от notebook 09  
**Что получим:** шесть входных путей

In [4]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports" / "tables"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
TRANSACTIONS_PATH = RAW_DIR / "transactions_train.csv"
ARTICLES_PATH = RAW_DIR / "articles.csv"
CUSTOMERS_PATH = RAW_DIR / "customers.csv"
for directory in [PROCESSED_DIR, MODEL_DIR, REPORT_DIR, ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CATBOOST_PATH = MODEL_DIR / "catboost_recommender.cbm"
TEST_TABLE_PATH = PROCESSED_DIR / "test_ranking_table.parquet"
FEATURES_PATH = MODEL_DIR / "feature_columns.json"
CATEGORIES_PATH = MODEL_DIR / "categorical_features.json"
POPULARITY_PATH = MODEL_DIR / "popular_items.json"
METADATA_PATH = MODEL_DIR / "model_metadata.json"
DEMO_OUTPUT_PATH = ARTIFACT_DIR / "batch_demo_recommendations.parquet"

### Проверка artifacts

**Что делаем:** проверяем модель, table и JSON  
**Зачем:** batch inference не должен незаметно переобучать отсутствующую модель  
**Что получим:** понятную ошибку с notebook 09

In [5]:
required_paths = [
    CATBOOST_PATH, TEST_TABLE_PATH, FEATURES_PATH,
    CATEGORIES_PATH, POPULARITY_PATH, METADATA_PATH,
]
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        f"Не найдены inference artifacts: {missing_paths}. "
        "Сначала выполните notebook 09_catboost_ranking_colab.ipynb."
    )

### Загрузка модели

**Что делаем:** читаем сохранённый CatBoost `.cbm`  
**Зачем:** метод `.fit()` здесь не вызывается  
**Что получим:** `batch_model`

In [6]:
batch_model = load_catboost_model(CATBOOST_PATH)
print("Loaded trees:", batch_model.tree_count_)

Loaded trees: 36


### Загрузка ranking table

**Что делаем:** читаем готовые test candidates и features  
**Зачем:** ALS/Content/candidate generation не повторяются  
**Что получим:** `batch_table`

In [7]:
batch_table = pd.read_parquet(TEST_TABLE_PATH)
print("Batch table:", batch_table.shape)
display(batch_table.head())

Batch table: (445123, 42)
                                         customer_id  ... target
0  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0
1  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0
2  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0
3  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0
4  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...      0

[5 rows x 42 columns]


### Загрузка config

**Что делаем:** читаем порядок features, categories и fallback items  
**Зачем:** inference должен повторять training schema  
**Что получим:** три небольших списка

In [8]:
feature_columns = load_json(FEATURES_PATH)
categorical_features = load_json(CATEGORIES_PATH)
popular_items = load_json(POPULARITY_PATH)
print("Features:", len(feature_columns))
print("Categories:", categorical_features)
print("Fallback items:", len(popular_items))

Features: 39
Categories: ['product_type_name', 'product_group_name', 'colour_group_name', 'department_name', 'section_name', 'garment_group_name']
Fallback items: 100


### Batch X

**Что делаем:** выбираем features и приводим categories к строкам  
**Зачем:** ID и target не передаются модели  
**Что получим:** `batch_x`

In [9]:
batch_x = batch_table[feature_columns].copy()
for column in categorical_features:
    batch_x[column] = batch_x[column].astype(str)
print("Batch X:", batch_x.shape)

Batch X: (445123, 39)


### Batch predict_proba

**Что делаем:** получаем scores сохранённой моделью  
**Зачем:** это inference без обучения  
**Что получим:** `batch_score`

In [10]:
batch_scored = batch_table[["customer_id", "article_id"]].copy()
batch_scored["batch_score"] = batch_model.predict_proba(batch_x)[:, 1]
display(batch_scored.head())

                                         customer_id  article_id  batch_score
0  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  0372860001     0.201056
1  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  0562245046     0.250631
2  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  0706016003     0.274873
3  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  0759871002     0.187841
4  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  0673396002     0.187606


### Сортировка и Top-12

**Что делаем:** сортируем scores внутри user и берём первые позиции  
**Зачем:** fallback будет отдельным шагом  
**Что получим:** `batch_top12`

In [11]:
batch_ranked = batch_scored.sort_values(
    ["customer_id", "batch_score", "article_id"],
    ascending=[True, False, True],
).drop_duplicates(["customer_id", "article_id"])
batch_top12 = batch_ranked.groupby(
    "customer_id", sort=False
).head(12)
print(batch_top12.groupby("customer_id").size().describe())

count    2000.0
mean       12.0
std         0.0
min        12.0
25%        12.0
50%        12.0
75%        12.0
max        12.0
dtype: float64


### Batch fallback

**Что делаем:** дополняем короткие lists сохранённой popularity  
**Зачем:** результат имеет тот же contract, что notebook 09  
**Что получим:** `batch_rows`

In [12]:
batch_rows = []
for customer_id, group in batch_top12.groupby("customer_id", sort=False):
    chosen = list(zip(group["article_id"], group["batch_score"]))
    chosen_ids = {article_id for article_id, _ in chosen}
    for article_id in popular_items:
        if article_id not in chosen_ids:
            chosen.append((article_id, 0.0))
            chosen_ids.add(article_id)
        if len(chosen) == 12:
            break
    for rank, (article_id, score) in enumerate(chosen[:12], start=1):
        batch_rows.append({
            "customer_id": customer_id,
            "article_id": article_id,
            "rank": rank,
            "score": float(score),
        })

### Сохранение demo batch

**Что делаем:** создаём DataFrame и сохраняем отдельный Parquet  
**Зачем:** основной `final_recommendations.parquet` из notebook 09 не перезаписывается  
**Что получим:** `batch_demo_recommendations.parquet`

In [13]:
batch_recommendations = pd.DataFrame(batch_rows)
saved_demo_path = save_recommendations(
    batch_recommendations,
    DEMO_OUTPUT_PATH,
)
print("Сохранено:", saved_demo_path)
print("Rows:", len(batch_recommendations))

Сохранено: <PROJECT_ROOT>/artifacts/batch_demo_recommendations.parquet
Rows: 24000


### Демонстрация API-файла

**Что делаем:** загружаем сохранённый Parquet проверенным loader  
**Зачем:** API использует такой же формат для lookup  
**Что получим:** несколько готовых рекомендаций

In [14]:
loaded_demo = load_recommendations(DEMO_OUTPUT_PATH)
example_users = loaded_demo["customer_id"].drop_duplicates().head(2)
display(loaded_demo[loaded_demo["customer_id"].isin(example_users)])

                                          customer_id  ...     score
0   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.464738
1   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.425352
2   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.403344
3   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.377075
4   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.373831
5   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.371906
6   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.367070
7   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.349636
8   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.344203
9   0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.337603
10  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.332008
11  0056238bf79efd3e95fc0ed4f4c01c018ff088544fa010...  ...  0.330356
12  005fad428070a5a317fbfacdfab69295a944879326605b...  ...  0.693468
13  005fad428070a5a317fbfacdfab692

### Metadata

**Что делаем:** показываем фактическую информацию сохранённой модели  
**Зачем:** endpoint `/model-info` возвращает этот объект  
**Что получим:** словарь model metadata

In [15]:
model_metadata = load_json(METADATA_PATH)
display(model_metadata)

{'architecture': 'Popularity + Personal History + ALS + Content-Based -> CatBoostClassifier', 'trained_at': '2026-08-07 22:52:25.533890+00:00', 'prediction_horizon_days': 7, 'recommendation_size': 12, 'evaluation_user_limit': 2000, 'final_test_metrics': {'model': 'CatBoost Hybrid', 'Recall@12': 0.017166666666666663, 'MAP@12': 0.00784229797979798, 'HitRate@12': 0.0185, 'Candidate Recall': 0.05734999999999999, 'users_evaluated': 2000, 'average_candidates': 222.5615, 'training_time': 2.001511959009804, 'inference_time': 0.1688599589979276, 'notes': 'CatBoostClassifier; common test cohort'}, 'final_test_window_start': '2019-12-25 00:00:00', 'final_test_window_end': '2019-12-31 00:00:00'}
